# 01 - Exploratory Data Analysis (EDA) | PM2.5 Forecasting Project
**Master 1 Mini Project | Academic Year 2026-2027**

This notebook executes the dataset audit, sentinel value cleaning, outlier detection, and correlation analysis in compliance with **Sections 3.1 and 5.1** of the project specification.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RAW_PATH = '../data/raw/combined_gee_pm25.csv'
CLEANED_PATH = '../data/processed/data_cleaned.csv'
SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')


## 1. Mandatory Dataset Audit (§3.1)
Auditing row counts, temporal range, sensor count, GEE IDs, and checking key-field duplicates.

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
print(f'Temporal coverage: {df_raw["date"].min()} to {df_raw["date"].max()}')
print(f'Unique air quality monitoring stations: {df_raw["sensor_name"].nunique()}')
print(f'Unique GEE spatial grid cells: {df_raw["id"].nunique()}')

# Duplicates verification
exact_dups = int(df_raw.duplicated().sum())
key_dups = int(df_raw.duplicated(subset=['date', 'latitude', 'longitude', 'sensor_name']).sum())
print(f'Exact duplicates: {exact_dups} | Key-field duplicates: {key_dups}')
df_raw.head(3)


## 2. Target Cleaning & Invalid Sentinel Removal (§5.1)
Remove records where PM2.5 is null or invalid sentinel values: $PM_{2.5} \le 0$ or $PM_{2.5} \ge 500\ \mu g/m^3$ **before** EDA and splitting.

In [ ]:
invalid_mask = df_raw['pm25'].isnull() | (df_raw['pm25'] <= 0) | (df_raw['pm25'] >= 500)
n_removed = int(invalid_mask.sum())
df_clean = df_raw[~invalid_mask].copy()
print(f'Removed {n_removed} invalid records ({n_removed / len(df_raw) * 100:.2f}%).')
print(f'Retained valid observations: {len(df_clean)}')
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv(CLEANED_PATH, index=False)


## 3. Target Distribution & WHO Threshold Analysis
Inspection of right-skewness and computing proportion of days exceeding $25\ \mu g/m^3$ and $50\ \mu g/m^3$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df_clean['pm25'], bins=60, kde=True, ax=axes[0], color='#1f77b4')
axes[0].axvline(25, color='orange', linestyle='--', label='WHO 24h Threshold (25 ug/m3)')
axes[0].axvline(50, color='red', linestyle='--', label='Severe Pollution (50 ug/m3)')
axes[0].set_title('Raw Daily PM2.5 (Right-Skewed)')
axes[0].set_xlim(0, 100)
axes[0].legend()

sns.histplot(np.log1p(df_clean['pm25']), bins=50, kde=True, ax=axes[1], color='#2a9d8f')
axes[1].set_title('Log-Transformed log1p(PM2.5) Target')
plt.tight_layout()
plt.show()

pct_25 = (df_clean['pm25'] > 25).mean() * 100
pct_50 = (df_clean['pm25'] > 50).mean() * 100
print(f'Proportion > 25 ug/m3: {pct_25:.2f}% | Proportion > 50 ug/m3: {pct_50:.2f}%')


## 4. Predictor Correlation Matrix (§5.1)
Spearman rank correlation heatmap of meteorology, satellite gases, and AOD.

In [ ]:
num_cols = ['pm25', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD']
corr = df_clean[num_cols].corr(method='spearman')
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Spearman Rank Correlation Matrix')
plt.tight_layout()
plt.show()
